# Semantic Chunking

Using embedding similarities between sentences to decide chunk boundaries to ensure every chunk is coherent and no data is cut off mid sentence.


In [3]:
import json
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough,
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage, HumanMessage

# Langchain Specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

load_dotenv()


c:\Users\Nilanjan\Documents\PoCs\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [4]:
# from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [5]:
## Initialize a simple Embedding model (no API Key needed!)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    # model_name="sentence-transformers/paraphrase-multilingual-MinLM-L12-v2",
    multi_process=True,
    show_progress=True,
    cache_folder="./../model_cache/",
    # model_kwargs = {"device": "cpu"}
    # model_kwargs = {"device": "gpu"}
    # model_kwargs = {"device": "cuda"}
)
embedding_model



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3477.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder='./../model_cache/', model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=True, show_progress=True)

In [22]:
# Step1: load and split sentences.
with open("./input_data.txt", "r") as f:
    data = f.read()

## Step 1: split into sentences
sentences = [sent.strip() for sent in data.split("\n") if sent.strip()]

## Step 2: Embed each sentence
embedding = embedding_model.embed_documents(sentences)

## Step 3: IInitialize parameters
threshold = 0.7 # control th etightness of the clusters
chunks = []
current_chunk = [sentences[0]]

## Step 4: Semantic grouping based on threshold

for i in range(1, len(sentences)):
    sim = cosine_similarity(
        [embedding[i-1]], 
        [embedding[i]]
    )[0][0]
    
    if sim >= threshold:
        current_chunk.append(sentences[i])
        print(f"Merging : {sentences[i]} with similarity {sim:.2f}\n")
    else:
        chunks.append(" ".join(current_chunk))
        print(f"Standalone Chunk {current_chunk} with similarity {sim:.2f}")
        current_chunk = [sentences[i]]
        print(f"Starting new chunk with: {sentences[i]}\n")

## Append the last chunk
print(" Semantic Chunks:")
for idx, chunk in enumerate(chunks):
    print(f"\nChunk {idx+1}: \n{chunk}")

Merging : Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone. with similarity 0.83

Standalone Chunk ['LangChain is a framework for building applications with LLMs.', 'Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'] with similarity 0.42
Starting new chunk with: You can create chains,| agents, memory, and retrievers.

Standalone Chunk ['You can create chains,| agents, memory, and retrievers.'] with similarity 0.03
Starting new chunk with: The Eiffel Tower is located in Paris.

Standalone Chunk ['The Eiffel Tower is located in Paris.'] with similarity 0.38
Starting new chunk with: France is a popular tourist destination.

 Semantic Chunks:

Chunk 1: 
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2: 
You can create chains,| agents, memory, and retrievers.

Chunk 3: 
The Eiffel Tower is lo

In [14]:
chunks

['LangChain is a framework for building applications with LLMs.',
 'LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.',
 'You can create chains,| agents, memory, and retrievers.',
 'The Eiffel Tower is located in Paris.']

In [11]:
data

'LangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains,| agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.'

In [12]:
sentences

['LangChain is a framework for building applications with LLMs.',
 'Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.',
 'You can create chains,| agents, memory, and retrievers.',
 'The Eiffel Tower is located in Paris.',
 'France is a popular tourist destination.']